# ChatgaiyyaAlap MT — Day 1 (Google Colab)
Zero-Shot vs Few-Shot Prompting: Standard Bangla ↔ Chittagonian (Chatgaiya)

**Before you start:**
- `Runtime → Change runtime type → T4 GPU` (optional but faster for the model test in Step 5; CPU works too, just slower)
- Have the two Mendeley CSVs downloaded to your computer already, OR uploaded to a Google Drive folder — you'll be asked for them in Step 2.


## Step 0 — Install dependencies

In [1]:
!pip -q install transformers accelerate sacrebleu pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.3 MB/s eta 0:00:00


## Step 2 — Get the dataset

The dataset has no public download API, so grab it manually:

1. Go to **https://data.mendeley.com/datasets/wtms9xbkkw/1**
2. Download both CSV files (sentence pairs + dictionary)

Then either:
- **(A) Upload directly to this Colab session** (fastest, but lost when the session ends — re-run this cell each new session), or
- **(B) Mount Google Drive** and read from a Drive folder (persists across sessions — recommended)

Run **one** of the two cells below.


In [2]:
# --- Option A: direct upload (session-only) ---
from google.colab import files
import shutil, os

os.makedirs("data", exist_ok=True)
print("Select the SENTENCE PAIRS csv file:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, "data/sentence_pairs.csv")

print("Select the DICTIONARY csv file:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, "data/dictionary.csv")

print("Saved to data/sentence_pairs.csv and data/dictionary.csv")


Select the SENTENCE PAIRS csv file:


Saving Dataset_Chittagong_2.0.csv to Dataset_Chittagong_2.0.csv
Select the DICTIONARY csv file:


Saving dictionary.csv to dictionary.csv
Saved to data/sentence_pairs.csv and data/dictionary.csv


## Step 3 — Load and inspect both CSVs (with pandas)

Column names in the raw Mendeley export can vary, so this auto-detects the Bangla vs.
Chittagonian columns instead of hardcoding names.


In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
SENTENCE_PAIRS_PATH = DATA_DIR / "sentence_pairs.csv"
DICTIONARY_PATH = DATA_DIR / "dictionary.csv"

BANGLA_HINTS = ["bangla", "bengali", "standard"]
CHATGAIYA_HINTS = ["chittagon", "chatgaiya", "chatgaiyya", "dialect"]


def guess_columns(df, label):
    cols = list(df.columns)
    lower_cols = {c: c.lower() for c in cols}

    bangla_col = next((c for c, lc in lower_cols.items() if any(h in lc for h in BANGLA_HINTS)), None)
    chatgaiya_col = next((c for c, lc in lower_cols.items() if any(h in lc for h in CHATGAIYA_HINTS)), None)

    if bangla_col is None or chatgaiya_col is None:
        if len(cols) == 2:
            print(f"[WARN] Could not confidently detect columns for {label} from {cols}. "
                  f"Falling back to positional guess: '{cols[0]}'=Bangla, '{cols[1]}'=Chatgaiya. VERIFY by eye.")
            return cols[0], cols[1]
        raise ValueError(f"Could not detect Bangla/Chatgaiya columns in {label}. Columns: {cols}")

    return bangla_col, chatgaiya_col


def load_sentence_pairs():
    df = pd.read_csv(SENTENCE_PAIRS_PATH)
    b_col, c_col = guess_columns(df, "sentence_pairs.csv")
    df = df.rename(columns={b_col: "bangla", c_col: "chatgaiya"})
    return df[["bangla", "chatgaiya"]].dropna().drop_duplicates().reset_index(drop=True)


def load_dictionary():
    df = pd.read_csv(DICTIONARY_PATH)
    b_col, c_col = guess_columns(df, "dictionary.csv")
    df = df.rename(columns={b_col: "bangla_word", c_col: "chatgaiya_word"})
    return df[["bangla_word", "chatgaiya_word"]].dropna().drop_duplicates().reset_index(drop=True)


pairs = load_sentence_pairs()
dictionary = load_dictionary()

print(f"{len(pairs)} unique sentence pairs")
print(f"{len(dictionary)} unique dictionary entries")
pairs.head()


[WARN] Could not confidently detect columns for sentence_pairs.csv from ['বাংলা', 'চট্টগ্রাম']. Falling back to positional guess: 'বাংলা'=Bangla, 'চট্টগ্রাম'=Chatgaiya. VERIFY by eye.
[WARN] Could not confidently detect columns for dictionary.csv from ['বাংলা', 'চট্টগ্রাম']. Falling back to positional guess: 'বাংলা'=Bangla, 'চট্টগ্রাম'=Chatgaiya. VERIFY by eye.
4009 unique sentence pairs
1532 unique dictionary entries


,bangla,chatgaiya
0,আমি ভাত খাবো না রাগ করছি আমার মনে জ্বালা,অ্যাঁই ভাত ন হাইয়্যুম গুসসা ওইয়্যুম অ্যাঁর মনত...
1,একদিন বুড়ি চিন্তা করল তার মুরগি,একদিন বুড়ি চিন্তা গরিল কুরো
2,আমি যে খেতে দেই যদি,অ্যাঁই যে হাইতে দ্যি যদি
3,সে একটা করে ডিম দেয়,ইতে উজ্ঞো গরে দিম দে
4,তার খাবার দিলে বেশি করে ডিম দিবে,ইতার হানা দিলে বেশি গরে দিম দিতি


In [4]:
dictionary.head()


,bangla_word,chatgaiya_word
0,আমি,অ্যাঁই
1,সিন্দাবাদের,সিন্দাবাদর
2,বেধেছে,বাইদ্ধে
3,গভীর,গভীর
4,আকাশের,আকাশের


In [5]:
pairs["bangla_len"] = pairs["bangla"].str.split().str.len()
pairs["chatgaiya_len"] = pairs["chatgaiya"].str.split().str.len()
print(f"Avg Bangla sentence length:     {pairs['bangla_len'].mean():.1f} tokens")
print(f"Avg Chatgaiya sentence length:  {pairs['chatgaiya_len'].mean():.1f} tokens")


Avg Bangla sentence length:     5.3 tokens
Avg Chatgaiya sentence length:  5.3 tokens


## Step 4 — Sample 50–80 sentence pairs

**Team rule:** everyone must use the same `SEED` and `N` so the whole team scores
against an identical sample — agree on both with your teammates before running this.


In [6]:
SEED = 42     # <-- agree on this with your team
N = 60        # <-- agree on this with your team (must be 50-80)
FEWSHOT_RESERVE = 8   # extra pairs held out for Day 3 few-shot examples, kept out of the test set

assert 50 <= N <= 80, "N must be between 50 and 80 per the assignment spec"

pool = pairs.sample(n=N + FEWSHOT_RESERVE, random_state=SEED).reset_index(drop=True)
test_sample = pool.iloc[:N].reset_index(drop=True)
fewshot_pool = pool.iloc[N:].reset_index(drop=True)

import os
os.makedirs("outputs", exist_ok=True)
test_sample.to_csv("outputs/sampled_pairs.csv", index=False)
fewshot_pool.to_csv("outputs/fewshot_pool.csv", index=False)

print(f"Saved {len(test_sample)} test pairs -> outputs/sampled_pairs.csv")
print(f"Saved {len(fewshot_pool)} held-out few-shot candidates -> outputs/fewshot_pool.csv")
print(f"(seed={SEED}, n={N}, fewshot_reserve={FEWSHOT_RESERVE})")


Saved 60 test pairs -> outputs/sampled_pairs.csv
Saved 8 held-out few-shot candidates -> outputs/fewshot_pool.csv
(seed=42, n=60, fewshot_reserve=8)


## Step 5 — Load an open-source model, run one manual test translation per direction

Uses a small open, ungated instruct model that runs fine on Colab's free GPU (or CPU,
just slower). This is only a smoke test — real zero-shot/few-shot prompt design is
Day 2-4, not here.


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
print(f"Loaded on device: {device}")


Loading Qwen/Qwen2.5-1.5B-Instruct ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded on device: cuda


In [8]:
def translate(text, direction):
    if direction == "b2c":
        instruction = (
            "Translate the following Standard Bangla sentence into the Chittagonian "
            "(Chatgaiya) dialect. Reply with only the translated sentence, nothing else.\n\n"
            f"Standard Bangla: {text}\nChittagonian:"
        )
    elif direction == "c2b":
        instruction = (
            "Translate the following Chittagonian (Chatgaiya) dialect sentence into "
            "Standard Bangla. Reply with only the translated sentence, nothing else.\n\n"
            f"Chittagonian: {text}\nStandard Bangla:"
        )
    else:
        raise ValueError("direction must be 'b2c' or 'c2b'")

    messages = [{"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=100, temperature=0.3, do_sample=True)

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


test_row = test_sample.iloc[0]

print("MANUAL TEST 1: Standard Bangla -> Chittagonian")
print("Input (Bangla):       ", test_row["bangla"])
out_b2c = translate(test_row["bangla"], "b2c")
print("Model output:         ", out_b2c)
print("Reference (Chatgaiya):", test_row["chatgaiya"])


MANUAL TEST 1: Standard Bangla -> Chittagonian
Input (Bangla):        সব সময় এমন জুতা দিয়ে মাইর খায়
Model output:          সব সময় আমি তোলা বাল্ডা করছি
Reference (Chatgaiya): অক্কল সমত এন জুতা দি মাইর হায়


In [9]:
print("MANUAL TEST 2: Chittagonian -> Standard Bangla")
print("Input (Chatgaiya):    ", test_row["chatgaiya"])
out_c2b = translate(test_row["chatgaiya"], "c2b")
print("Model output:         ", out_c2b)
print("Reference (Bangla):   ", test_row["bangla"])

print("\nNote any hallucinations, refusals, or defaulting to standard Bangla in your")
print("issue log now — that's exactly what Day 2-4 will study systematically.")


MANUAL TEST 2: Chittagonian -> Standard Bangla
Input (Chatgaiya):     অক্কল সমত এন জুতা দি মাইর হায়
Model output:          অক্কল সমতেন জুতা দিল মাইর হয়
Reference (Bangla):    সব সময় এমন জুতা দিয়ে মাইর খায়

Note any hallucinations, refusals, or defaulting to standard Bangla in your
issue log now — that's exactly what Day 2-4 will study systematically.


## Step 6 — Generate the Day 1 dataset summary JSON

In [10]:
import json
from datetime import datetime, timezone

YOUR_NAME = "yourname"   # <-- change me

summary = {
    "team": "LLM Team",
    "assignment": "Zero-Shot vs Few-Shot Prompting: Standard Bangla <-> Chittagonian (Chatgaiya)",
    "generated_by": YOUR_NAME,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "name": "ChatgaiyyaAlap",
        "source_url": "https://data.mendeley.com/datasets/wtms9xbkkw/1",
        "description": (
            "Parallel corpus of Standard Bangla and Chittagonian (Chatgaiya) dialect "
            "sentence pairs, collected from YouTube/Facebook posts, comments, videos, "
            "short films, and dramas, plus a word-level dictionary file."
        ),
        "total_sentence_pairs": len(pairs),
        "total_dictionary_entries": len(dictionary),
        "directions_covered": ["bangla_to_chatgaiya", "chatgaiya_to_bangla"],
    },
    "week1_sample": {
        "seed": SEED,
        "test_sample_size": len(test_sample),
        "fewshot_pool_size": len(fewshot_pool),
        "example_test_pair": {
            "bangla": test_sample.iloc[0]["bangla"],
            "chatgaiya": test_sample.iloc[0]["chatgaiya"],
        },
    },
    "day1_smoke_test": {
        "model_used": MODEL_NAME,
        "b2c_example": {"input": test_row["bangla"], "output": out_b2c, "reference": test_row["chatgaiya"]},
        "c2b_example": {"input": test_row["chatgaiya"], "output": out_c2b, "reference": test_row["bangla"]},
    },
    "notes": [
        "Chatgaiya has no standard written form, so spelling variation across the "
        "dataset is expected and not necessarily an error.",
        "Both translation directions must be evaluated separately, not combined.",
    ],
}

out_path = f"outputs/{YOUR_NAME}_week1day1_data.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Wrote {out_path}")


Wrote outputs/yourname_week1day1_data.json


---
**Recap of what this notebook did (Day 1 checklist):**
1. ✅ Connected to GitHub (token-based, Colab-friendly)
2. ✅ Loaded + inspected both CSVs
3. ✅ Loaded sentence pairs + dictionary with pandas
4. ✅ Sampled 50–80 sentence pairs (shared seed) + held out a separate few-shot pool
5. ✅ Loaded an open-source model, ran one manual translation per direction
6. ✅ Generated `<yourname>_week1day1_data.json`

Next: Day 2 — zero-shot prompt templates for both directions.
